In [1]:
import pandas as pd
import re
from difflib import get_close_matches
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import VarianceThreshold
import numpy as np

In [2]:
# Read the genetic data
genetic_data = pd.read_csv(r"C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\GeneMarkers-95 varieties 1.csv")

genetic_data.info
genetic_data.head(10)

,taglo_id,ATLANTIC_124776,ESCORT_159228,ARGOS_120931,ALTURAS_329540,ELMUNDO_835520,NADINE_241950,VIOLETQUEEN_3345402,DEODARA_513721,MEMPHIS_2279529,...,DIAMANT_153502,INNOVATOR_234757,TRIPLE7_3347283,SPUNTA_283648,ANTI_118190,CARRERA_142554,FORTUS_3279866,SMART_1883446,SALINERO_253609,FABULA_171173
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,17,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,22,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,39,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,48,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,49,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,54,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [3]:
genetic_data.isnull().sum()

taglo_id           0
ATLANTIC_124776    0
ESCORT_159228      0
ARGOS_120931       0
ALTURAS_329540     0
                  ..
CARRERA_142554     0
FORTUS_3279866     0
SMART_1883446      0
SALINERO_253609    0
FABULA_171173      0
Length: 95, dtype: int64

In [4]:
# Clean column names
genetic_columns_clean = [c.strip().upper().replace(" ", "").replace(".", "") for c in genetic_data.columns]

# Assign cleaned names back to DataFrame
genetic_data.columns = genetic_columns_clean

# Now you can view columns
print(genetic_data.columns)


Index(['TAGLO_ID', 'ATLANTIC_124776', 'ESCORT_159228', 'ARGOS_120931',
       'ALTURAS_329540', 'ELMUNDO_835520', 'NADINE_241950',
       'VIOLETQUEEN_3345402', 'DEODARA_513721', 'MEMPHIS_2279529',
       'AGRIA_125336', 'PAYETTERUSSET_4617676', 'IVORYRUSSET_1360551',
       'SABABA_4959615', 'CECILE_147140', 'FRISIA_163592', 'CARDYMA_3345873',
       'RICKEYRUSSET_4640884', 'ANIVIA_4446779', 'AVARNA_1630052',
       'DONALD_158741', 'JAERLA_231308', 'ARIZONA_3056017',
       'SARPOMIRA_1629377', 'MARILYN_1632173', 'DESIREE_139717',
       'MUSE_6159099', 'ADORA_123802', 'TAURUS_1883495', 'BERBER_121293',
       'RANGERRUSSET_265488', 'BINTJE_126680', 'FENWAYRED_4055844',
       'KONDOR_248484', 'SAGITTA_1459411', 'CLEARWATERR_2721777',
       'NICOLA_248062', 'TETONRUSSET_3029162', 'FESTIEN_172619',
       'HANSA_180216', 'PEEWEERUSSET_3347291', 'PRINCEOFORANGE_416115',
       'SUPERIOR_1477892', 'VANGOGH_294181', 'CHALLENGER_1883438',
       'JENNIFER_3221728', 'PARELLA_2983716', 'CH

In [5]:
merged_aroma_sensory= pd.read_csv(r'C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\merged_aroma_sensory.csv')
print(merged_aroma_sensory)

         Variety  1-Nonanol (Floral, waxy)  \
0          ADORA                       0.0   
1          AGRIA                       0.0   
2       ALOUETTE                       0.0   
3         ALTHEA                       0.0   
4        ALTURAS                       0.0   
..           ...                       ...   
89  TETON RUSSET                       0.0   
90       TRIPLE7                       0.0   
91      VAN GOGH                       0.0   
92     VE 71-105                       0.0   
93  VIOLET QUEEN                       0.0   

    2,3-Butanedione (Butter, creamy, sweet)  \
0                                       0.0   
1                                       0.0   
2                                       0.0   
3                                       0.0   
4                                       0.0   
..                                      ...   
89                                      0.0   
90                                      0.0   
91                      

In [6]:
import re

def normalize_variety(name):
    if not isinstance(name, str):
        return ""
    name = name.upper()
    name = name.replace(".", "")
    name = re.sub(r'\bR\b', '', name)
    name = re.sub(r'RUSSET', '', name)
    name = name.replace(" ", "")
    name = name.strip()
    name = name.split('_')[0]
    return name

# Normalize genetic data index
genetic_data_t = genetic_data.set_index('TAGLO_ID').T
genetic_data_t.index = genetic_data_t.index.map(normalize_variety)

# Same corrections dictionary
name_corrections = {
    'CLEARWATERR': 'CLEARWATER',
    'ALVERSTONER': 'ALVERSTONE'
}

# Apply corrections to genetic data index as well
genetic_data_t.index = genetic_data_t.index.to_series().replace(name_corrections)

# Normalize aroma sensory data
merged_aroma_sensory['Variety_norm'] = merged_aroma_sensory['Variety'].apply(normalize_variety)
merged_aroma_sensory['Variety_norm'] = merged_aroma_sensory['Variety_norm'].replace(name_corrections)
merged_aroma_sensory = merged_aroma_sensory.set_index('Variety_norm')

# Now merge on corrected index
combined_df = merged_aroma_sensory.merge(genetic_data_t, left_index=True, right_index=True, how='inner')

print(f"Combined shape after correction: {combined_df.shape}")

# Check missing varieties again
aroma_varieties = set(merged_aroma_sensory.index)
genetic_varieties = set(genetic_data_t.index)

missing_in_genetic = aroma_varieties - genetic_varieties
missing_in_aroma = genetic_varieties - aroma_varieties

print(f"Varieties missing in genetic data: {missing_in_genetic}")
print(f"Varieties missing in aroma/sensory data: {missing_in_aroma}")
combined_df.head(5)

Combined shape after correction: (94, 262461)
Varieties missing in genetic data: set()
Varieties missing in aroma/sensory data: set()


,Variety,"1-Nonanol (Floral, waxy)","2,3-Butanedione (Butter, creamy, sweet)","Benzaldehyde (Almond, cherry, fruity)","Butanal, 3-methyl- (Fruity, malty, chocolate)","Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)","Decanal (Citrus, floral)","Dimethyl trisulfide (Garlic, onion, sulfurous)","Furfural (Sweet, almond, caramel)","Hexanal (Green, grassy)",...,564850,564851,564852,564853,564854,564857,564858,564859,564860,564861
ADORA,ADORA,0.0,0.0,17.984870,15.836435,17.417208,18.203338,0.00000,0.0,0.00000,...,2,4,3,1,0,0,0,0,0,2
AGRIA,AGRIA,0.0,0.0,14.859493,14.154847,15.947810,14.499810,17.21077,0.0,0.00000,...,0,4,1,0,0,0,0,0,0,0
ALOUETTE,ALOUETTE,0.0,0.0,15.205705,15.262178,14.678352,17.462371,0.00000,0.0,0.00000,...,3,4,3,2,0,0,0,0,0,3
ALTHEA,ALTHEA,0.0,0.0,19.535081,14.984406,17.704453,21.158234,0.00000,0.0,0.00000,...,0,4,0,0,0,0,0,0,0,0
ALTURAS,ALTURAS,0.0,0.0,20.293776,14.482897,14.257052,16.590508,0.00000,0.0,17.24572,...,1,4,2,1,0,0,0,0,1,1


# Feature Reduction on X (Genetic Markers)

Variance thresholding (remove near-constant features)

In [7]:
flavor_cols = [
    'Sweet', 'Metallic Flavour', 'Bitter Flavour',
    'Earthy Flavour', 'Sour Flavour', 'Fresh Flavour', 'Sweet Flavour',
    'Root/ Vegetable Flavour', 'Farmyard (grass/hay) flavour',
    'Bitter Aftertaste', 'Sour Aftertaste', 'Sweet Aftertaste'
]
exclude_cols = flavor_cols + ['Variety'] + [ 
    '1-Nonanol (Floral, waxy)', '2,3-Butanedione (Butter, creamy, sweet)', 'Benzaldehyde (Almond, cherry, fruity)',
    'Butanal, 3-methyl- (Fruity, malty, chocolate)', 'Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)',
    'Decanal (Citrus, floral)', 'Dimethyl trisulfide (Garlic, onion, sulfurous)', 'Furfural (Sweet, almond, caramel)',
    'Hexanal (Green, grassy)', 'Hexanoic acid (Fatty, cheesy, sweaty)', 'Methional (Cooked potato, sulfurous)', 
    'Octanal (Citrus, fruity)', 'Pentanal (Green, fatty, pungent)'
]

X = combined_df.drop(columns=exclude_cols, errors='ignore')
y = combined_df[flavor_cols]

print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (94, 262435), y shape: (94, 12)


In [8]:

# 1) Ensure numeric values (coerce non-numeric to NaN if any)
X = X.apply(pd.to_numeric, errors='coerce')

# 2) Make ALL column names strings (prevents the TypeError)
X.columns = X.columns.astype(str)

# 3) (Optional but useful) drop columns that are entirely NaN
# X = X.dropna(axis=1, how='all')

# 4) Variance threshold
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)  # fit first so we can keep names
support = selector.get_support()

# 5) Keep index + names
X_reduced = pd.DataFrame(
    selector.transform(X),
    index=X.index,
    columns=X.columns[support]
)

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Reduced X shape: {X_reduced.shape}")


X shape: (94, 262435), y shape: (94, 12)
Reduced X shape: (94, 261111)


Use feature selection based on correlation with target (univariate filter)

In [9]:

# For each flavor, select top features, or just once for general
selector = SelectKBest(score_func=f_regression, k=500)  # top 500 features

X_selected = selector.fit_transform(X, y['Sweet'].fillna(0))  # example flavor to select features
print(f"Shape after SelectKBest: {X_selected.shape}")


Shape after SelectKBest: (94, 500)


<!-- Lasso regression -->

In [10]:
for flavor in flavor_cols:
    y_flavor = y[flavor].dropna()
    mask = y[flavor].notna()
    
    X_train = X_selected[mask.values]
    y_train = y_flavor.values
    
    pls = PLSRegression(n_components=5)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pls, X_train, y_train, cv=cv, scoring='r2')
    
    print(f"{flavor} PLS CV R^2: {np.mean(scores):.3f} ± {np.std(scores):.3f}")


Sweet PLS CV R^2: 0.503 ± 0.294
Metallic Flavour PLS CV R^2: 0.369 ± 0.199
Bitter Flavour PLS CV R^2: 0.395 ± 0.171
Earthy Flavour PLS CV R^2: 0.499 ± 0.146
Sour Flavour PLS CV R^2: 0.436 ± 0.242
Fresh Flavour PLS CV R^2: 0.567 ± 0.131
Sweet Flavour PLS CV R^2: 0.501 ± 0.454
Root/ Vegetable Flavour PLS CV R^2: 0.546 ± 0.180
Farmyard (grass/hay) flavour PLS CV R^2: 0.478 ± 0.152
Bitter Aftertaste PLS CV R^2: 0.346 ± 0.145
Sour Aftertaste PLS CV R^2: 0.450 ± 0.246
Sweet Aftertaste PLS CV R^2: 0.320 ± 0.620


# Run Elastic Net & Random Forest on reduced dataset

In [11]:


# --- 1) Feature selection once (using 'Sweet' as the proxy target) ---
k = min(500, X.shape[1])  # don't exceed number of features
selector = SelectKBest(score_func=f_regression, k=k)
selector.fit(X, y['Sweet'].fillna(0))

support = selector.get_support()
selected_cols = X.columns[support]

# Keep index + names so .loc works with your mask
X_selected = pd.DataFrame(
    selector.transform(X),
    index=X.index,
    columns=selected_cols
)

print(f"Shape after SelectKBest: {X_selected.shape}")

# --- 2) Models (scale for ElasticNet) ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'ElasticNet': make_pipeline(StandardScaler(), ElasticNet(max_iter=5000, random_state=42)),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
}

# --- 3) Loop over flavors ---
for flavor in flavor_cols:
    mask = y[flavor].notna()            # rows with a label for this flavor
    X_train = X_selected.loc[mask]      # works now (index-aligned)
    y_train = y.loc[mask, flavor]

    print(f"\nResults for {flavor}:")
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')
        print(f"{name} CV R^2: {np.mean(scores):.3f} ± {np.std(scores):.3f}")


Shape after SelectKBest: (94, 500)

Results for Sweet:
ElasticNet CV R^2: 0.551 ± 0.072
RandomForest CV R^2: 0.782 ± 0.137

Results for Metallic Flavour:
ElasticNet CV R^2: 0.727 ± 0.084
RandomForest CV R^2: 0.886 ± 0.055

Results for Bitter Flavour:
ElasticNet CV R^2: 0.727 ± 0.104
RandomForest CV R^2: 0.862 ± 0.073

Results for Earthy Flavour:
ElasticNet CV R^2: 0.721 ± 0.054
RandomForest CV R^2: 0.931 ± 0.028

Results for Sour Flavour:
ElasticNet CV R^2: 0.704 ± 0.063
RandomForest CV R^2: 0.918 ± 0.036

Results for Fresh Flavour:
ElasticNet CV R^2: 0.778 ± 0.106
RandomForest CV R^2: 0.962 ± 0.018

Results for Sweet Flavour:
ElasticNet CV R^2: 0.665 ± 0.042
RandomForest CV R^2: 0.869 ± 0.020

Results for Root/ Vegetable Flavour:
ElasticNet CV R^2: 0.771 ± 0.053
RandomForest CV R^2: 0.958 ± 0.021

Results for Farmyard (grass/hay) flavour:
ElasticNet CV R^2: 0.706 ± 0.082
RandomForest CV R^2: 0.847 ± 0.080

Results for Bitter Aftertaste:
ElasticNet CV R^2: 0.731 ± 0.091
RandomForest CV

These results suggest strong predictability between your selected volatile features and flavor scores.

The 500 top features selected by SelectKBest seem to retain enough signal for both linear (ElasticNet) and non-linear (RandomForest) models.

High R² in CV (with low ±SD) suggests the model generalizes well — but 94 samples is still relatively small, so there’s always a risk of optimistic estimates if features are correlated.



# for being sure we check overfitting,Permutation tests,Feature importanc and smaller feature sets

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

cv = KFold(n_splits=5, shuffle=True, random_state=42)

n_features_to_test = [500, 200, 100]  # test feature set sizes

for flavor in flavor_cols:
    mask = y[flavor].notna()
    y_train = y.loc[mask, flavor]
    
    print(f"\n=== {flavor} ===")
    
    for k in n_features_to_test:
        # Top-k feature selection using the same flavor's target
        selector = SelectKBest(score_func=f_regression, k=min(k, X.shape[1]))
        X_k = selector.fit_transform(X.loc[mask], y_train)
        selected_cols = X.columns[selector.get_support()]
        
        # 1) Fit on full data to get training R²
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf.fit(X_k, y_train)
        train_r2 = r2_score(y_train, rf.predict(X_k))
        
        # 2) Cross-validation R²
        cv_r2 = cross_val_score(rf, X_k, y_train, cv=cv, scoring="r2", n_jobs=-1)
        
        # 3) Permutation test: shuffle target
        y_shuffled = y_train.sample(frac=1.0, random_state=42).reset_index(drop=True)
        cv_r2_perm = cross_val_score(rf, X_k, y_shuffled, cv=cv, scoring="r2", n_jobs=-1)
        
        print(f"Top-{k} features:")
        print(f"  Train R²: {train_r2:.3f}")
        print(f"  CV R²: {np.mean(cv_r2):.3f} ± {np.std(cv_r2):.3f}")
        print(f"  Permuted CV R²: {np.mean(cv_r2_perm):.3f} ± {np.std(cv_r2_perm):.3f}")
        
        # 4) Feature importance (only show top 5)
        importances = pd.Series(rf.feature_importances_, index=selected_cols)
        top5 = importances.sort_values(ascending=False).head(5)
        print("  Top 5 features:\n", top5.to_string())



=== Sweet ===
Top-500 features:
  Train R²: 0.981
  CV R²: 0.782 ± 0.137
  Permuted CV R²: -0.003 ± 0.414
  Top 5 features:
 Intensity of Flavour    0.803774
299105                  0.016678
163063                  0.014004
105214                  0.010647
299021                  0.009776
Top-200 features:
  Train R²: 0.980
  CV R²: 0.778 ± 0.158
  Permuted CV R²: -0.150 ± 0.249
  Top 5 features:
 Intensity of Flavour    0.805416
163063                  0.018881
299105                  0.016690
298984                  0.012192
299021                  0.011755
Top-100 features:
  Train R²: 0.983
  CV R²: 0.769 ± 0.202
  Permuted CV R²: -0.338 ± 0.328
  Top 5 features:
 Intensity of Flavour    0.815741
163063                  0.022548
299105                  0.018470
25874                   0.015175
485140                  0.013344

=== Metallic Flavour ===
Top-500 features:
  Train R²: 0.986
  CV R²: 0.898 ± 0.064
  Permuted CV R²: -0.320 ± 0.171
  Top 5 features:
 Intensity of Flavour

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# 1) Remove "Intensity of Flavour" from X
X_no_intensity = X.drop(columns=["Intensity of Flavour"], errors="ignore")

cv = KFold(n_splits=5, shuffle=True, random_state=42)
k = min(200, X_no_intensity.shape[1])  # keep feature set modest for speed

for flavor in flavor_cols:
    mask = y[flavor].notna()
    Xf = X_no_intensity.loc[mask]
    yf = y.loc[mask, flavor]

    # --- Select top-k features ---
    selector = SelectKBest(score_func=f_regression, k=k)
    Xk = selector.fit_transform(Xf, yf)
    selected_cols = Xf.columns[selector.get_support()]

    # --- RandomForest ---
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(Xk, yf)

    train_r2 = r2_score(yf, rf.predict(Xk))
    cv_r2 = cross_val_score(rf, Xk, yf, cv=cv, scoring="r2", n_jobs=-1)

    # --- Feature importances ---
    importances = pd.Series(rf.feature_importances_, index=selected_cols)
    top5 = importances.sort_values(ascending=False).head(5)

    print(f"\n=== {flavor} ===")
    print(f"Train R²: {train_r2:.3f} | CV R²: {np.mean(cv_r2):.3f} ± {np.std(cv_r2):.3f}")
    print("Top 5 features:\n", top5.to_string())



=== Sweet ===
Train R²: 0.945 | CV R²: 0.420 ± 0.418
Top 5 features:
 299105    0.179132
475518    0.076483
162964    0.040878
299077    0.039585
162999    0.039216

=== Metallic Flavour ===
Train R²: 0.911 | CV R²: 0.282 ± 0.120
Top 5 features:
 475481    0.076203
428792    0.064146
475551    0.046070
457062    0.045412
48675     0.042945

=== Bitter Flavour ===
Train R²: 0.911 | CV R²: 0.389 ± 0.100
Top 5 features:
 373299    0.042868
166707    0.038792
398375    0.033471
475481    0.033110
152264    0.032984

=== Earthy Flavour ===
Train R²: 0.913 | CV R²: 0.364 ± 0.118
Top 5 features:
 340351    0.042728
398375    0.037590
48675     0.031605
299275    0.030650
253306    0.029888

=== Sour Flavour ===
Train R²: 0.916 | CV R²: 0.382 ± 0.108
Top 5 features:
 475481    0.067544
475551    0.053889
475438    0.034340
260210    0.033766
398375    0.031870

=== Fresh Flavour ===
Train R²: 0.920 | CV R²: 0.412 ± 0.273
Top 5 features:
 149077    0.084851
152264    0.051922
162964    0.04623

<!-- Filter genetic markers by variance -->